# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Show basic dataset information
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Published: {md.datePublished}, Version: {md.version}")
print(f"License: {md.license}")
print(f"Cite as: {getattr(md, 'citeAs', '')}")

## 2. Data Overview
Review available record sets, their `@id`s, and the fields/columns within. The `mlcroissant` library exposes record sets and their structure via the metadata.

In [ ]:
# Explore the available record sets and fields by @id
record_sets = getattr(dataset.metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in the metadata. Please check the schema.")
else:
    for rs in record_sets:
        print(f"Record set @id: {getattr(rs, '@id', None)}")
        print(f"  Name: {getattr(rs, 'name', None)}")
        if hasattr(rs, 'field'):
            print("  Fields (with @id):")
            for fld in rs.field:
                print(f"    - {getattr(fld, '@id', None)} | {getattr(fld, 'name', None)} | {getattr(fld, 'dataType', None)}")
        if hasattr(rs, 'column'):
            print("  Columns (with @id):")
            for col in rs.column:
                print(f"    - {getattr(col, '@id', None)} | {getattr(col, 'name', None)} | {getattr(col, 'dataType', None)}")
        print()

## 3. Data Extraction
Load data from each record set into a DataFrame. For each, use the record set and field `@id`s as referenced in the previous section.

In [ ]:
# Use the @id of the relevant record sets to extract data

# 1. Identify available record set IDs
record_set_ids = [(rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)) for rs in getattr(dataset.metadata, 'recordSet', [])]
print(f"Available record sets: {record_set_ids}")

# 2. Load each record set as a DataFrame
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for record set {rs_id}: {df.shape[0]} rows, {df.shape[1]} columns")

# 3. Print columns of the first record set for orientation, and show some rows
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"\nColumns in {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data for summary statistics.


In [ ]:
import numpy as np

# Choose a numeric field (by @id or column name, which is the same as field @id)
# Inspect available columns - here, as an example, we pick one containing 'coverage' or 'MW' (molecular weight), typical for mass spectrometry
df = dataframes[selected_record_set_id]
numeric_field_candidates = [col for col in df.columns if ('coverage' in col.lower() or 'weight' in col.lower() or 'intensity' in col.lower() or df[col].dtype in [np.float64, np.int64])]

if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    print(f"Using numeric field for filtering: {numeric_field}")
    threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 10
    # Remove rows with missing/invalid numeric values
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Choose a group field
    group_field_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}, showing means of numeric columns:")
        display(grouped_df.head())
else:
    print("No numeric field automatically detected. Please inspect the columns and select a numeric field for analysis.")

## 5. Visualization
Visualize data distributions or relationships between numeric and categorical fields in the dataset. Below are some example plots.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_candidates:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # If group field exists, boxplot
    if group_field_candidates:
        plt.figure(figsize=(10,5))
        top_cats = df[group_field].value_counts().index[:5]
        sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_cats)])
        plt.title(f"{numeric_field} by {group_field} (top 5)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded the mass spectrometry dataset using `mlcroissant`, reviewed its record sets (using Croissant `@id` references), extracted data into DataFrames, applied simple preprocessing and exploratory analysis steps (including normalization and grouping), and visualized key numeric distributions.

For further analysis, refer to the fields discussed in the metadata, and use proper `@id` references for robust and reproducible workflows.

Happy exploring!